In [2]:
import argparse
import json
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ── Config ─────────────────────────────────────────────────────────────────────
CATEGORICAL_COLS = ['ContractType', 'PaymentMethod', 'InternetService', 'TechSupport', 'OnlineSecurity']
DROP_COLS        = ['Unnamed: 0', 'CustomerID']

# ── Main ───────────────────────────────────────────────────────────────────────
def main(data_path: str):

    # 1. Load data
    print(f"Loading data from: {data_path}")
    df = pd.read_csv(data_path)
    print(f"  Shape: {df.shape}")
    print(f"  Churn rate: {df['Churn'].mean()*100:.1f}%")

    # 2. Preprocess
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
    df = pd.get_dummies(df, columns=CATEGORICAL_COLS, drop_first=True)

    X = df.drop('Churn', axis=1)
    y = df['Churn']

    # 3. Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)

    # 4. Scale
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    # 5. Train
    print("Training Random Forest ...")
    model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    model.fit(X_train_s, y_train)

    # 6. Evaluate
    y_pred  = model.predict(X_test_s)
    y_proba = model.predict_proba(X_test_s)[:, 1]
    results = {
        'accuracy':  round(accuracy_score(y_test, y_pred), 4),
        'precision': round(precision_score(y_test, y_pred), 4),
        'recall':    round(recall_score(y_test, y_pred), 4),
        'f1':        round(f1_score(y_test, y_pred), 4),
        'roc_auc':   round(roc_auc_score(y_test, y_proba), 4),
    }
    print("  Results:", results)

    # 7. Save with joblib
    joblib.dump(model,              'model.joblib')
    joblib.dump(scaler,             'scaler.joblib')
    joblib.dump(X.columns.tolist(), 'feature_names.joblib')
    with open('model_results.json', 'w') as f:
        json.dump(results, f, indent=2)

    print("Saved: model.joblib, scaler.joblib, feature_names.joblib, model_results.json")


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--data', default='Customer_Churn.xls')
    args = parser.parse_args(args=[])
    main(args.data)


Loading data from: Customer_Churn.xls
  Shape: (2000, 14)
  Churn rate: 48.9%
Training Random Forest ...
  Results: {'accuracy': 0.6025, 'precision': 0.5989, 'recall': 0.5714, 'f1': 0.5849, 'roc_auc': 0.6407}
Saved: model.joblib, scaler.joblib, feature_names.joblib, model_results.json
